# VoxConverse Dataset Explorer et Analyseur de Qualité

Notebook complet pour l'exploration du dataset VoxConverse avec analyse de qualité des données et widgets interactifs.

## Table des matières
1. **Configuration et imports** - Setup de l'environnement
2. **Variables globales configurables** - Paramètres personnalisables  
3. **Analyse de qualité des données** - Statistiques et métriques
4. **Génération de dataset paramétrable** - Création flexible de datasets
5. **Widget interactif de visualisation** - Interface dynamique
6. **Lecture audio et analyse détaillée** - Exploration approfondie

In [ ]:
# 1. Configuration et imports
import sys
import os
sys.path.append('/home/dev/Speaker-diarization-/src')

import torch
import torch.nn.functional as F
import torchaudio
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from IPython.display import Audio, display, HTML
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed
import warnings
warnings.filterwarnings("ignore")

# Import du dataset VoxConverse
from voxconverse_dataset import VoxConverseDataset, create_voxconverse_dataloaders

# Configuration matplotlib
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("✅ Imports réussis!")
print(f"🔥 CUDA disponible: {torch.cuda.is_available()}")
print(f"🐍 Python: {sys.version.split()[0]}")
print(f"🔊 PyTorch: {torch.__version__}")
print(f"🎵 Torchaudio: {torchaudio.__version__}")

## 2. Variables Globales Configurables

Ces variables contrôlent tous les aspects de la génération et visualisation du dataset :

In [ ]:
# ===== VARIABLES GLOBALES CONFIGURABLES =====

# 🎛️ Paramètres de segmentation du dataset
SEGMENT_DURATION =10.0 # Durée des segments en secondes (PERSONNALISABLE!)
HOP_DURATION = 2.0          # Hop entre segments en secondes
SAMPLE_RATE = 16000         # Fréquence d'échantillonnage
N_MELS = 80                 # Nombre de bandes mel

# 🎯 Segment à visualiser dans le widget (PERSONNALISABLE!)
SELECTED_SEGMENT_IDX = 0    # Index du segment à afficher (changez cette valeur!)

# 📊 Paramètres d'affichage
DISPLAY_TIME_RANGE = (0, SEGMENT_DURATION) # Plage temporelle à afficher (start, end) en secondes
FIGSIZE_LARGE = (16, 12)    # Taille des figures
FIGSIZE_MEDIUM = (12, 8)    # Taille moyenne
COLOR_PALETTE = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

# 🔍 Paramètres d'analyse qualité
MAX_SEGMENTS_TO_ANALYZE = 1000  # Limite pour l'analyse rapide
QUALITY_THRESHOLD_VAD = 0.1     # Seuil minimum de VAD pour un "bon" segment
QUALITY_THRESHOLD_OSD = 0.05    # Seuil maximum d'OSD pour éviter trop d'overlap

print("🎛️ Variables globales configurées:")
print(f"   📏 Durée segment: {SEGMENT_DURATION}s")
print(f"   🎯 Segment sélectionné: #{SELECTED_SEGMENT_IDX}")
print(f"   📊 Plage d'affichage: {DISPLAY_TIME_RANGE[0]}-{DISPLAY_TIME_RANGE[1]}s")
print(f"   🔍 Max segments à analyser: {MAX_SEGMENTS_TO_ANALYZE}")
print(f"")
print("💡 Pour changer la durée des segments, modifiez SEGMENT_DURATION")
print("💡 Pour visualiser un autre segment, modifiez SELECTED_SEGMENT_IDX")

## 3. Chargement et Analyse Initiale du Dataset

In [ ]:
# Chargement du dataset avec les paramètres globaux
print("🔄 Chargement du dataset VoxConverse...")
print(f"   Utilisation de segments de {SEGMENT_DURATION}s avec hop de {HOP_DURATION}s")

# Créer le dataset avec nos paramètres configurables
dataset = VoxConverseDataset(
    split='dev',
    segment_duration=SEGMENT_DURATION,  # Utilise notre variable globale!
    hop_duration=HOP_DURATION,
    sample_rate=SAMPLE_RATE,
    n_mels=N_MELS,
    min_speaker_duration=0.5
)

print(f"✅ Dataset chargé avec succès!")
print(f"   📦 Nombre total de segments: {len(dataset)}")
print(f"   📏 Durée par segment: {SEGMENT_DURATION}s")
print(f"   🎵 Fréquence d'échantillonnage: {SAMPLE_RATE} Hz")
print(f"   🔊 Bandes mel: {N_MELS}")

# Vérifier qu'on a des données
if len(dataset) == 0:
    print("❌ ERREUR: Aucun segment trouvé dans le dataset!")
else:
    print(f"🎯 Premier segment disponible: index 0 à {len(dataset)-1}")
    print(f"🎯 Segment sélectionné pour visualisation: #{SELECTED_SEGMENT_IDX}")

## 4. Analyse de Qualité des Données

Analysons la distribution et la qualité des segments VAD, OSD et VCN :

In [ ]:
def analyze_dataset_quality(dataset, max_segments=MAX_SEGMENTS_TO_ANALYZE):
    """Analyse complète de la qualité du dataset"""
    
    print(f"🔍 Analyse de qualité sur {min(len(dataset), max_segments)} segments...")
    
    # Collecter les statistiques
    stats = {
        'vad_frames': [],
        'osd_frames': [], 
        'vcn_frames': [],
        'total_frames': [],
        'has_speech': [],
        'has_overlap': [],
        'has_voice_change': [],
        'conv_idx': [],
        'start_time': []
    }
    
    # Analyser un échantillon de segments
    sample_size = min(len(dataset), max_segments)
    for i in range(sample_size):
        if i % 100 == 0:
            print(f"   Progression: {i}/{sample_size}")
            
        segment = dataset.segments[i]
        stats['vad_frames'].append(segment.get('vad_frames', 0))
        stats['osd_frames'].append(segment.get('osd_frames', 0))
        stats['vcn_frames'].append(segment.get('vcn_frames', 0))
        stats['total_frames'].append(int(SEGMENT_DURATION / 0.02))  # 20ms frames
        stats['has_speech'].append(segment.get('vad_frames', 0) > 0)
        stats['has_overlap'].append(segment.get('has_overlap', False))
        stats['has_voice_change'].append(segment.get('has_voice_change', False))
        stats['conv_idx'].append(segment.get('conv_idx', -1))
        stats['start_time'].append(segment.get('start_time', 0))
    
    # Convertir en DataFrame pour l'analyse
    df = pd.DataFrame(stats)
    
    # Calculer les ratios
    df['vad_ratio'] = df['vad_frames'] / df['total_frames']
    df['osd_ratio'] = df['osd_frames'] / df['total_frames'] 
    df['vcn_ratio'] = df['vcn_frames'] / df['total_frames']
    
    return df

# Analyser la qualité
df_quality = analyze_dataset_quality(dataset)

print(f"✅ Analyse terminée sur {len(df_quality)} segments")

In [ ]:
# Visualisations de qualité
fig, axes = plt.subplots(2, 3, figsize=FIGSIZE_LARGE)
fig.suptitle('📊 Analyse de Qualité du Dataset VoxConverse', fontsize=16, fontweight='bold')

# 1. Distribution des ratios VAD
axes[0,0].hist(df_quality['vad_ratio'], bins=50, alpha=0.7, color=COLOR_PALETTE[0], edgecolor='black')
axes[0,0].axvline(QUALITY_THRESHOLD_VAD, color='red', linestyle='--', label=f'Seuil qualité: {QUALITY_THRESHOLD_VAD}')
axes[0,0].set_title('Distribution VAD (Voice Activity)')
axes[0,0].set_xlabel('Ratio de frames avec voix')
axes[0,0].set_ylabel('Nombre de segments')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# 2. Distribution des ratios OSD  
axes[0,1].hist(df_quality['osd_ratio'], bins=50, alpha=0.7, color=COLOR_PALETTE[1], edgecolor='black')
axes[0,1].axvline(QUALITY_THRESHOLD_OSD, color='red', linestyle='--', label=f'Seuil qualité: {QUALITY_THRESHOLD_OSD}')
axes[0,1].set_title('Distribution OSD (Overlap Speech)')
axes[0,1].set_xlabel('Ratio de frames avec overlap')
axes[0,1].set_ylabel('Nombre de segments')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# 3. Distribution des ratios VCN
axes[0,2].hist(df_quality['vcn_ratio'], bins=50, alpha=0.7, color=COLOR_PALETTE[2], edgecolor='black')
axes[0,2].set_title('Distribution VCN (Voice Change)')
axes[0,2].set_xlabel('Ratio de frames avec changement')
axes[0,2].set_ylabel('Nombre de segments')
axes[0,2].grid(True, alpha=0.3)

# 4. Comparaison VAD vs OSD
axes[1,0].scatter(df_quality['vad_ratio'], df_quality['osd_ratio'], alpha=0.5, color=COLOR_PALETTE[3])
axes[1,0].set_title('VAD vs OSD')
axes[1,0].set_xlabel('Ratio VAD')
axes[1,0].set_ylabel('Ratio OSD')
axes[1,0].grid(True, alpha=0.3)

# 5. Segments par conversation
conv_counts = df_quality['conv_idx'].value_counts().head(20)
axes[1,1].bar(range(len(conv_counts)), conv_counts.values, color=COLOR_PALETTE[4])
axes[1,1].set_title('Segments par conversation (Top 20)')
axes[1,1].set_xlabel('Conversations')
axes[1,1].set_ylabel('Nombre de segments')
axes[1,1].grid(True, alpha=0.3)

# 6. Pourcentages globaux
percentages = [
    df_quality['has_speech'].sum() / len(df_quality) * 100,
    df_quality['has_overlap'].sum() / len(df_quality) * 100, 
    df_quality['has_voice_change'].sum() / len(df_quality) * 100
]
labels = ['VAD\n(Speech)', 'OSD\n(Overlap)', 'VCN\n(Change)']
bars = axes[1,2].bar(labels, percentages, color=COLOR_PALETTE[:3])
axes[1,2].set_title('Pourcentage de segments avec activité')
axes[1,2].set_ylabel('Pourcentage (%)')
axes[1,2].grid(True, alpha=0.3)

# Ajouter les valeurs sur les barres
for bar, pct in zip(bars, percentages):
    height = bar.get_height()
    axes[1,2].text(bar.get_x() + bar.get_width()/2., height + 1,
                   f'{pct:.1f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Statistiques résumées
print(f"\n📈 STATISTIQUES DE QUALITÉ")
print(f"="*50)
print(f"📊 Segments analysés: {len(df_quality)}")
print(f"📏 Durée par segment: {SEGMENT_DURATION}s")
print(f"")
print(f"🎤 VAD (Voice Activity Detection):")
print(f"   Segments avec parole: {df_quality['has_speech'].sum()} ({df_quality['has_speech'].mean()*100:.1f}%)")
print(f"   Ratio moyen VAD: {df_quality['vad_ratio'].mean():.3f} ± {df_quality['vad_ratio'].std():.3f}")
print(f"   Segments de qualité (>{QUALITY_THRESHOLD_VAD}): {(df_quality['vad_ratio'] > QUALITY_THRESHOLD_VAD).sum()}")
print(f"")
print(f"🗣️  OSD (Overlap Speech Detection):")
print(f"   Segments avec overlap: {df_quality['has_overlap'].sum()} ({df_quality['has_overlap'].mean()*100:.1f}%)")
print(f"   Ratio moyen OSD: {df_quality['osd_ratio'].mean():.3f} ± {df_quality['osd_ratio'].std():.3f}")
print(f"   Segments peu d'overlap (<{QUALITY_THRESHOLD_OSD}): {(df_quality['osd_ratio'] < QUALITY_THRESHOLD_OSD).sum()}")
print(f"")
print(f"🔄 VCN (Voice Change Detection):")
print(f"   Segments avec changements: {df_quality['has_voice_change'].sum()} ({df_quality['has_voice_change'].mean()*100:.1f}%)")
print(f"   Ratio moyen VCN: {df_quality['vcn_ratio'].mean():.3f} ± {df_quality['vcn_ratio'].std():.3f}")


## 5. Génération de Dataset Paramétrable

Fonction pour créer un dataset filtré avec des critères de qualité :

In [ ]:
def create_custom_dataset(
    segment_duration=SEGMENT_DURATION,  # Utilise notre variable globale!
    hop_duration=HOP_DURATION,
    min_vad_ratio=0.1,
    max_osd_ratio=0.3,
    require_voice_changes=False,
    max_segments=None
):
    """
    Crée un dataset personnalisé avec des critères de qualité.
    
    Args:
        segment_duration: Durée des segments (utilise la variable globale par défaut)
        hop_duration: Hop entre segments
        min_vad_ratio: Ratio minimum de VAD requis
        max_osd_ratio: Ratio maximum d'OSD accepté
        require_voice_changes: Si True, ne garde que les segments avec changements
        max_segments: Nombre max de segments (None = tous)
    """
    
    print(f"🔧 Création d'un dataset personnalisé...")
    print(f"   📏 Durée segments: {segment_duration}s (hop: {hop_duration}s)")
    print(f"   🎤 VAD minimum: {min_vad_ratio:.2f}")
    print(f"   🗣️  OSD maximum: {max_osd_ratio:.2f}")
    print(f"   🔄 Changements requis: {require_voice_changes}")
    print(f"   📦 Limite segments: {max_segments or 'Aucune'}")
    
    # Créer le dataset avec les nouveaux paramètres
    custom_dataset = VoxConverseDataset(
        split='dev',
        segment_duration=segment_duration,
        hop_duration=hop_duration,
        sample_rate=SAMPLE_RATE,
        n_mels=N_MELS,
        min_speaker_duration=0.5
    )
    
    print(f"📊 Dataset brut créé: {len(custom_dataset)} segments")
    
    # Filtrer selon les critères de qualité
    filtered_segments = []
    
    for i, segment in enumerate(custom_dataset.segments):
        if max_segments and len(filtered_segments) >= max_segments:
            break
            
        total_frames = int(segment_duration / 0.02)  # 20ms frames
        vad_ratio = segment.get('vad_frames', 0) / total_frames
        osd_ratio = segment.get('osd_frames', 0) / total_frames
        has_changes = segment.get('has_voice_change', False)
        
        # Appliquer les filtres
        if vad_ratio >= min_vad_ratio and osd_ratio <= max_osd_ratio:
            if not require_voice_changes or has_changes:
                filtered_segments.append(segment)
    
    # Remplacer les segments du dataset
    custom_dataset.segments = filtered_segments
    
    print(f"✅ Dataset filtré créé: {len(custom_dataset)} segments")
    print(f"   📉 Réduction: {(1 - len(custom_dataset) / len(custom_dataset.segments if hasattr(custom_dataset, 'segments') else []))*100:.1f}%")
    
    return custom_dataset

# Exemple: créer un dataset de haute qualité avec notre durée configurée
print(f"🎯 Test avec SEGMENT_DURATION = {SEGMENT_DURATION}s")
high_quality_dataset = create_custom_dataset(
    segment_duration=SEGMENT_DURATION,  # Utilise notre variable!
    min_vad_ratio=0.3,      # Plus strict sur la parole
    max_osd_ratio=0.1,      # Moins d'overlap autorisé
    require_voice_changes=False,
    max_segments=500        # Limite pour le test
)

print(f"\n💡 Pour changer la durée, modifiez SEGMENT_DURATION en haut du notebook!")
print(f"💡 Actuellement configuré à {SEGMENT_DURATION}s")

## 6. Widget Dynamique de Visualisation

Interface interactive pour explorer les segments avec mel spectrogramme, VAD, OSD, VCN et lecture audio :

In [ ]:
def visualize_segment(dataset, segment_idx=SELECTED_SEGMENT_IDX):
    """
    Visualise un segment avec mel spectrogramme, VAD, OSD, VCN et lecture audio.
    Utilise la variable globale SELECTED_SEGMENT_IDX par défaut.
    """
    
    if segment_idx >= len(dataset):
        print(f"❌ Segment {segment_idx} n'existe pas (max: {len(dataset)-1})")
        return
    
    print(f"🎯 Visualisation du segment #{segment_idx}")
    print(f"   📏 Durée configurée: {SEGMENT_DURATION}s")
    
    # Obtenir le segment
    sample = dataset[segment_idx]
    segment_info = dataset.segments[segment_idx]
    
    # Données pour la visualisation
    mel_features = sample['features'].numpy()  # [n_mels, time]
    vad_labels = sample['vad_labels'].numpy()  # [time]
    osd_labels = sample['osd_labels'].numpy()  # [time]
    vcn_labels = sample['vcn_labels'].numpy()  # [time]
    audio = sample['audio'].numpy()
    
    # Configuration temporelle
    time_frames = mel_features.shape[1]
    time_axis = np.linspace(0, SEGMENT_DURATION, time_frames)
    audio_time = np.linspace(0, SEGMENT_DURATION, len(audio))
    
    # Créer la figure avec 5 sous-graphiques
    fig, axes = plt.subplots(5, 1, figsize=(16, 14))
    fig.suptitle(f'🎵 Segment #{segment_idx} - Conv {segment_info["conv_idx"]} - '
                 f't={segment_info["start_time"]:.1f}s-{segment_info["end_time"]:.1f}s', 
                 fontsize=16, fontweight='bold')
    
    # 1. Mel Spectrogramme
    im = axes[0].imshow(mel_features, aspect='auto', origin='lower', 
                        extent=[0, SEGMENT_DURATION, 0, N_MELS],
                        cmap='viridis')
    axes[0].set_title('🎼 Mel Spectrogramme (Log)', fontweight='bold')
    axes[0].set_ylabel('Bandes Mel')
    axes[0].set_xlim(DISPLAY_TIME_RANGE)
    plt.colorbar(im, ax=axes[0], label='Log Magnitude')
    
    # 2. Signal audio
    axes[1].plot(audio_time, audio, color='blue', alpha=0.7, linewidth=0.5)
    axes[1].set_title('🔊 Signal Audio', fontweight='bold')
    axes[1].set_ylabel('Amplitude')
    axes[1].set_xlim(DISPLAY_TIME_RANGE)
    axes[1].grid(True, alpha=0.3)
    
    # 3. VAD (Voice Activity Detection)
    axes[2].fill_between(time_axis, 0, vad_labels, alpha=0.7, color='green', label='VAD')
    axes[2].plot(time_axis, vad_labels, color='darkgreen', linewidth=2)
    axes[2].set_title('🎤 VAD - Voice Activity Detection', fontweight='bold')
    axes[2].set_ylabel('Activation')
    axes[2].set_xlim(DISPLAY_TIME_RANGE)
    axes[2].set_ylim(-0.1, 1.1)
    axes[2].grid(True, alpha=0.3)
    axes[2].legend()
    
    # 4. OSD (Overlap Speech Detection)  
    axes[3].fill_between(time_axis, 0, osd_labels, alpha=0.7, color='orange', label='OSD')
    axes[3].plot(time_axis, osd_labels, color='darkorange', linewidth=2)
    axes[3].set_title('🗣️  OSD - Overlap Speech Detection', fontweight='bold')
    axes[3].set_ylabel('Activation')
    axes[3].set_xlim(DISPLAY_TIME_RANGE)
    axes[3].set_ylim(-0.1, 1.1)
    axes[3].grid(True, alpha=0.3)
    axes[3].legend()
    
    # 5. VCN (Voice Change Detection)
    axes[4].fill_between(time_axis, 0, vcn_labels, alpha=0.7, color='red', label='VCN')
    axes[4].plot(time_axis, vcn_labels, color='darkred', linewidth=2)
    axes[4].set_title('🔄 VCN - Voice Change Detection', fontweight='bold')
    axes[4].set_ylabel('Activation')
    axes[4].set_xlabel('Temps (secondes)')
    axes[4].set_xlim(DISPLAY_TIME_RANGE)
    axes[4].set_ylim(-0.1, 1.1)
    axes[4].grid(True, alpha=0.3)
    axes[4].legend()
    
    plt.tight_layout()
    plt.show()
    
    # Statistiques du segment
    print(f"\n📊 STATISTIQUES DU SEGMENT #{segment_idx}")
    print(f"="*50)
    print(f"🎵 Audio: {len(audio)} échantillons à {SAMPLE_RATE}Hz")
    print(f"🎼 Mel: {mel_features.shape} (bandes × temps)")
    print(f"⏱️  Frames: {time_frames} frames de 20ms")
    print(f"")
    print(f"🎤 VAD: {np.sum(vad_labels > 0)}/{len(vad_labels)} frames actives ({np.mean(vad_labels)*100:.1f}%)")
    print(f"🗣️  OSD: {np.sum(osd_labels > 0)}/{len(osd_labels)} frames overlap ({np.mean(osd_labels)*100:.1f}%)")
    print(f"🔄 VCN: {np.sum(vcn_labels > 0)}/{len(vcn_labels)} frames changement ({np.mean(vcn_labels)*100:.1f}%)")
    print(f"")
    print(f"👥 Speakers: {segment_info.get('speaker_ids', [])}")
    print(f"📍 Position: Conversation {segment_info['conv_idx']}, {segment_info['start_time']:.1f}s-{segment_info['end_time']:.1f}s")
    
    # Lecture audio
    print(f"\n🎧 LECTURE AUDIO:")
    display(Audio(audio, rate=SAMPLE_RATE))
    
    return sample

# Visualiser le segment sélectionné
print(f"🎯 Visualisation du segment configuré: #{SELECTED_SEGMENT_IDX}")
print(f"💡 Pour changer le segment, modifiez SELECTED_SEGMENT_IDX en haut!")

if len(dataset) > SELECTED_SEGMENT_IDX:
    current_sample = visualize_segment(dataset, SELECTED_SEGMENT_IDX)
else:
    print(f"❌ Segment {SELECTED_SEGMENT_IDX} non disponible. Maximum: {len(dataset)-1}")

## 7. Widget Interactif avec Contrôles

Widget avec sliders pour explorer dynamiquement différents segments :

In [ ]:
def create_interactive_widget(dataset):
    """Crée un widget interactif pour explorer les segments"""
    
    def update_visualization(segment_idx, show_audio_player, time_start, time_end):
        """Fonction appelée quand les contrôles changent"""
        
        if segment_idx >= len(dataset):
            print(f"❌ Segment {segment_idx} n'existe pas (max: {len(dataset)-1})")
            return
        
        # Obtenir le segment
        sample = dataset[segment_idx]
        segment_info = dataset.segments[segment_idx]
        
        # Données
        mel_features = sample['features'].numpy()
        vad_labels = sample['vad_labels'].numpy()
        osd_labels = sample['osd_labels'].numpy()
        vcn_labels = sample['vcn_labels'].numpy()
        audio = sample['audio'].numpy()
        
        # Configuration temporelle
        time_frames = mel_features.shape[1]
        time_axis = np.linspace(0, SEGMENT_DURATION, time_frames)
        audio_time = np.linspace(0, SEGMENT_DURATION, len(audio))
        
        # Créer la figure
        fig, axes = plt.subplots(5, 1, figsize=(16, 12))
        fig.suptitle(f'🎵 Segment #{segment_idx} - Conv {segment_info["conv_idx"]} - '
                     f't={segment_info["start_time"]:.1f}s-{segment_info["end_time"]:.1f}s', 
                     fontsize=14, fontweight='bold')
        
        # 1. Mel Spectrogramme
        im = axes[0].imshow(mel_features, aspect='auto', origin='lower',
                            extent=[0, SEGMENT_DURATION, 0, N_MELS],
                            cmap='viridis')
        axes[0].set_title('🎼 Mel Spectrogramme', fontweight='bold')
        axes[0].set_ylabel('Bandes Mel')
        axes[0].set_xlim(time_start, time_end)
        plt.colorbar(im, ax=axes[0], shrink=0.8)
        
        # 2. Signal audio
        axes[1].plot(audio_time, audio, color='blue', alpha=0.7, linewidth=0.8)
        axes[1].set_title('🔊 Signal Audio', fontweight='bold')
        axes[1].set_ylabel('Amplitude')
        axes[1].set_xlim(time_start, time_end)
        axes[1].grid(True, alpha=0.3)
        
        # 3. VAD
        axes[2].fill_between(time_axis, 0, vad_labels, alpha=0.7, color='green')
        axes[2].plot(time_axis, vad_labels, color='darkgreen', linewidth=2)
        axes[2].set_title('🎤 VAD - Voice Activity', fontweight='bold')
        axes[2].set_ylabel('Activation')
        axes[2].set_xlim(time_start, time_end)
        axes[2].set_ylim(-0.1, 1.1)
        axes[2].grid(True, alpha=0.3)
        
        # 4. OSD
        axes[3].fill_between(time_axis, 0, osd_labels, alpha=0.7, color='orange')
        axes[3].plot(time_axis, osd_labels, color='darkorange', linewidth=2)
        axes[3].set_title('🗣️  OSD - Overlap Speech', fontweight='bold')
        axes[3].set_ylabel('Activation')
        axes[3].set_xlim(time_start, time_end)
        axes[3].set_ylim(-0.1, 1.1)
        axes[3].grid(True, alpha=0.3)
        
        # 5. VCN
        axes[4].fill_between(time_axis, 0, vcn_labels, alpha=0.7, color='red')
        axes[4].plot(time_axis, vcn_labels, color='darkred', linewidth=2)
        axes[4].set_title('🔄 VCN - Voice Change', fontweight='bold')
        axes[4].set_ylabel('Activation')
        axes[4].set_xlabel('Temps (secondes)')
        axes[4].set_xlim(time_start, time_end)
        axes[4].set_ylim(-0.1, 1.1)
        axes[4].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Statistiques
        print(f"📊 Segment #{segment_idx} | Conv {segment_info['conv_idx']} | {segment_info['start_time']:.1f}s-{segment_info['end_time']:.1f}s")
        print(f"🎤 VAD: {np.sum(vad_labels > 0)}/{len(vad_labels)} ({np.mean(vad_labels)*100:.1f}%) | "
              f"🗣️  OSD: {np.sum(osd_labels > 0)}/{len(osd_labels)} ({np.mean(osd_labels)*100:.1f}%) | "
              f"🔄 VCN: {np.sum(vcn_labels > 0)}/{len(vcn_labels)} ({np.mean(vcn_labels)*100:.1f}%)")
        
        # Lecture audio si demandée
        if show_audio_player:
            print("🎧 Lecture audio:")
            display(Audio(audio, rate=SAMPLE_RATE))
    
    # Créer les contrôles interactifs
    max_segments = len(dataset) - 1
    
    segment_slider = widgets.IntSlider(
        value=SELECTED_SEGMENT_IDX,
        min=0,
        max=max_segments,
        description='Segment:',
        style={'description_width': 'initial'}
    )
    
    audio_checkbox = widgets.Checkbox(
        value=True,
        description='Lecture audio',
        style={'description_width': 'initial'}
    )
    
    time_start_slider = widgets.FloatSlider(
        value=DISPLAY_TIME_RANGE[0],
        min=0,
        max=SEGMENT_DURATION,
        step=0.1,
        description='Début (s):',
        style={'description_width': 'initial'}
    )
    
    time_end_slider = widgets.FloatSlider(
        value=DISPLAY_TIME_RANGE[1],
        min=0,
        max=SEGMENT_DURATION,
        step=0.1,
        description='Fin (s):',
        style={'description_width': 'initial'}
    )
    
    # Créer le widget interactif
    interactive_widget = interactive(
        update_visualization,
        segment_idx=segment_slider,
        show_audio_player=audio_checkbox,
        time_start=time_start_slider,
        time_end=time_end_slider
    )
    
    return interactive_widget

# Créer et afficher le widget interactif
print("🎛️ Widget interactif pour explorer les segments:")
print(f"   📦 {len(dataset)} segments disponibles")
print(f"   🎯 Segment initial: #{SELECTED_SEGMENT_IDX}")
print(f"   📏 Durée: {SEGMENT_DURATION}s par segment")

interactive_widget = create_interactive_widget(dataset)
display(interactive_widget)

## 8. Fonction de Comparaison Multi-Segments

Comparer plusieurs segments côte à côte :

In [ ]:
def compare_segments(dataset, segment_indices=[0, 1, 2], metrics=['VAD', 'OSD', 'VCN']):
    """
    Compare plusieurs segments côte à côte.
    
    Args:
        dataset: Dataset VoxConverse
        segment_indices: Liste des indices de segments à comparer
        metrics: Métriques à afficher ['VAD', 'OSD', 'VCN', 'MEL', 'AUDIO']
    """
    
    n_segments = len(segment_indices)
    n_metrics = len(metrics)
    
    # Vérifier que les segments existent
    valid_indices = [idx for idx in segment_indices if idx < len(dataset)]
    if not valid_indices:
        print("❌ Aucun segment valide à comparer")
        return
    
    print(f"🔍 Comparaison de {len(valid_indices)} segments: {valid_indices}")
    
    # Créer la figure
    fig, axes = plt.subplots(n_metrics, n_segments, figsize=(5*n_segments, 3*n_metrics))
    if n_segments == 1:
        axes = axes.reshape(-1, 1)
    if n_metrics == 1:
        axes = axes.reshape(1, -1)
    
    fig.suptitle(f'🔍 Comparaison Multi-Segments', fontsize=16, fontweight='bold')
    
    for col, seg_idx in enumerate(valid_indices):
        # Obtenir les données du segment
        sample = dataset[seg_idx]
        segment_info = dataset.segments[seg_idx]
        
        mel_features = sample['features'].numpy()
        vad_labels = sample['vad_labels'].numpy()
        osd_labels = sample['osd_labels'].numpy()
        vcn_labels = sample['vcn_labels'].numpy()
        audio = sample['audio'].numpy()
        
        time_frames = mel_features.shape[1]
        time_axis = np.linspace(0, SEGMENT_DURATION, time_frames)
        audio_time = np.linspace(0, SEGMENT_DURATION, len(audio))
        
        for row, metric in enumerate(metrics):
            ax = axes[row, col]
            
            if metric == 'MEL':
                # Mel spectrogramme
                im = ax.imshow(mel_features, aspect='auto', origin='lower',
                              extent=[0, SEGMENT_DURATION, 0, N_MELS],
                              cmap='viridis')
                ax.set_title(f'🎼 Seg #{seg_idx} - Mel')
                if col == 0:
                    ax.set_ylabel('Bandes Mel')
                
            elif metric == 'AUDIO':
                # Signal audio
                ax.plot(audio_time, audio, color='blue', alpha=0.7, linewidth=0.5)
                ax.set_title(f'🔊 Seg #{seg_idx} - Audio')
                if col == 0:
                    ax.set_ylabel('Amplitude')
                ax.grid(True, alpha=0.3)
                
            elif metric == 'VAD':
                # VAD
                ax.fill_between(time_axis, 0, vad_labels, alpha=0.7, color='green')
                ax.plot(time_axis, vad_labels, color='darkgreen', linewidth=2)
                ax.set_title(f'🎤 Seg #{seg_idx} - VAD ({np.mean(vad_labels)*100:.1f}%)')
                ax.set_ylim(-0.1, 1.1)
                if col == 0:
                    ax.set_ylabel('Activation')
                ax.grid(True, alpha=0.3)
                
            elif metric == 'OSD':
                # OSD
                ax.fill_between(time_axis, 0, osd_labels, alpha=0.7, color='orange')
                ax.plot(time_axis, osd_labels, color='darkorange', linewidth=2)
                ax.set_title(f'🗣️  Seg #{seg_idx} - OSD ({np.mean(osd_labels)*100:.1f}%)')
                ax.set_ylim(-0.1, 1.1)
                if col == 0:
                    ax.set_ylabel('Activation')
                ax.grid(True, alpha=0.3)
                
            elif metric == 'VCN':
                # VCN
                ax.fill_between(time_axis, 0, vcn_labels, alpha=0.7, color='red')
                ax.plot(time_axis, vcn_labels, color='darkred', linewidth=2)
                ax.set_title(f'🔄 Seg #{seg_idx} - VCN ({np.mean(vcn_labels)*100:.1f}%)')
                ax.set_ylim(-0.1, 1.1)
                if col == 0:
                    ax.set_ylabel('Activation')
                ax.grid(True, alpha=0.3)
            
            # Configuration de l'axe x
            if row == n_metrics - 1:  # Dernière ligne
                ax.set_xlabel('Temps (s)')
            ax.set_xlim(0, SEGMENT_DURATION)
    
    plt.tight_layout()
    plt.show()
    
    # Tableau de comparaison
    comparison_data = []
    for seg_idx in valid_indices:
        sample = dataset[seg_idx]
        segment_info = dataset.segments[seg_idx]
        
        vad_labels = sample['vad_labels'].numpy()
        osd_labels = sample['osd_labels'].numpy()
        vcn_labels = sample['vcn_labels'].numpy()
        
        comparison_data.append({
            'Segment': seg_idx,
            'Conversation': segment_info['conv_idx'],
            'Temps': f"{segment_info['start_time']:.1f}s-{segment_info['end_time']:.1f}s",
            'VAD (%)': f"{np.mean(vad_labels)*100:.1f}",
            'OSD (%)': f"{np.mean(osd_labels)*100:.1f}",
            'VCN (%)': f"{np.mean(vcn_labels)*100:.1f}",
            'Speakers': str(segment_info.get('speaker_ids', []))[:50]
        })
    
    df_comparison = pd.DataFrame(comparison_data)
    print(f"\n📊 TABLEAU DE COMPARAISON:")
    print(df_comparison.to_string(index=False))
    
    return df_comparison

# Exemple de comparaison avec des segments intéressants
print("🔍 Comparaison de segments avec différents profils:")

# Trouver des segments avec différentes caractéristiques
segments_high_vad = [i for i in range(min(50, len(dataset))) if dataset.segments[i].get('vad_frames', 0) > 150]
segments_with_overlap = [i for i in range(min(50, len(dataset))) if dataset.segments[i].get('osd_frames', 0) > 10]
segments_with_changes = [i for i in range(min(50, len(dataset))) if dataset.segments[i].get('vcn_frames', 0) > 5]

# Sélectionner des segments représentatifs
selected_for_comparison = []
if segments_high_vad:
    selected_for_comparison.append(segments_high_vad[0])
if segments_with_overlap:
    selected_for_comparison.append(segments_with_overlap[0])
if segments_with_changes:
    selected_for_comparison.append(segments_with_changes[0])

# Si pas assez de segments spéciaux, utiliser les premiers
while len(selected_for_comparison) < 3 and len(selected_for_comparison) < len(dataset):
    selected_for_comparison.append(len(selected_for_comparison))

if selected_for_comparison:
    comparison_result = compare_segments(
        dataset, 
        segment_indices=selected_for_comparison[:3],
        metrics=['MEL', 'VAD', 'OSD', 'VCN']
    )
else:
    print("❌ Pas assez de segments pour la comparaison")

## 9. Export et Sauvegarde

Fonctions pour sauvegarder les analyses et exporter des segments :

In [ ]:
def export_segment_analysis(dataset, segment_idx, output_dir='./exports'):
    """
    Exporte l'analyse complète d'un segment (audio, données, visualisations).
    """
    
    import os
    from datetime import datetime
    import pickle
    
    # Créer le dossier d'export
    os.makedirs(output_dir, exist_ok=True)
    
    if segment_idx >= len(dataset):
        print(f"❌ Segment {segment_idx} n'existe pas")
        return
    
    print(f"💾 Export du segment #{segment_idx}...")
    
    # Obtenir les données
    sample = dataset[segment_idx]
    segment_info = dataset.segments[segment_idx]
    
    # Nom de base pour les fichiers
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_name = f"segment_{segment_idx}_conv_{segment_info['conv_idx']}_{timestamp}"
    
    # 1. Sauvegarder l'audio
    audio_path = os.path.join(output_dir, f"{base_name}.wav")
    torchaudio.save(audio_path, sample['audio'].unsqueeze(0), SAMPLE_RATE)
    print(f"   🎵 Audio sauvé: {audio_path}")
    
    # 2. Sauvegarder les données en pickle
    data_to_save = {
        'segment_idx': segment_idx,
        'segment_info': segment_info,
        'mel_features': sample['features'].numpy(),
        'vad_labels': sample['vad_labels'].numpy(),
        'osd_labels': sample['osd_labels'].numpy(), 
        'vcn_labels': sample['vcn_labels'].numpy(),
        'audio': sample['audio'].numpy(),
        'sample_rate': SAMPLE_RATE,
        'segment_duration': SEGMENT_DURATION,
        'export_timestamp': timestamp
    }
    
    data_path = os.path.join(output_dir, f"{base_name}_data.pkl")
    with open(data_path, 'wb') as f:
        pickle.dump(data_to_save, f)
    print(f"   📊 Données sauvées: {data_path}")
    
    # 3. Créer et sauvegarder la visualisation
    sample_vis = visualize_segment(dataset, segment_idx)
    
    # Sauvegarder la figure actuelle
    viz_path = os.path.join(output_dir, f"{base_name}_visualization.png")
    plt.savefig(viz_path, dpi=300, bbox_inches='tight')
    print(f"   📈 Visualisation sauvée: {viz_path}")
    
    # 4. Créer un rapport texte
    report_path = os.path.join(output_dir, f"{base_name}_report.txt")
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write(f"RAPPORT D'ANALYSE - SEGMENT #{segment_idx}\\n")
        f.write(f"="*50 + "\\n")
        f.write(f"Date d'export: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\\n")
        f.write(f"Configuration: Durée={SEGMENT_DURATION}s, Sample_rate={SAMPLE_RATE}Hz\\n\\n")
        
        f.write(f"INFORMATIONS SEGMENT:\\n")
        f.write(f"  Index: {segment_idx}\\n")
        f.write(f"  Conversation: {segment_info['conv_idx']}\\n")
        f.write(f"  Temps: {segment_info['start_time']:.1f}s - {segment_info['end_time']:.1f}s\\n")
        f.write(f"  Speakers: {segment_info.get('speaker_ids', [])}\\n\\n")
        
        f.write(f"STATISTIQUES DÉTECTION:\\n")
        vad_pct = np.mean(sample['vad_labels'].numpy()) * 100
        osd_pct = np.mean(sample['osd_labels'].numpy()) * 100
        vcn_pct = np.mean(sample['vcn_labels'].numpy()) * 100
        
        f.write(f"  VAD (Voice Activity): {vad_pct:.1f}% des frames\\n")
        f.write(f"  OSD (Overlap Speech): {osd_pct:.1f}% des frames\\n")
        f.write(f"  VCN (Voice Change): {vcn_pct:.1f}% des frames\\n\\n")
        
        f.write(f"QUALITÉ:\\n")
        if vad_pct > QUALITY_THRESHOLD_VAD * 100:
            f.write(f"  ✅ VAD: Bonne activité vocale ({vad_pct:.1f}% > {QUALITY_THRESHOLD_VAD*100}%)\\n")
        else:
            f.write(f"  ⚠️  VAD: Faible activité vocale ({vad_pct:.1f}% < {QUALITY_THRESHOLD_VAD*100}%)\\n")
            
        if osd_pct <= QUALITY_THRESHOLD_OSD * 100:
            f.write(f"  ✅ OSD: Peu d'overlap ({osd_pct:.1f}% <= {QUALITY_THRESHOLD_OSD*100}%)\\n")
        else:
            f.write(f"  ⚠️  OSD: Beaucoup d'overlap ({osd_pct:.1f}% > {QUALITY_THRESHOLD_OSD*100}%)\\n")
        
        f.write(f"\\nFICHIERS GÉNÉRÉS:\\n")
        f.write(f"  - Audio: {base_name}.wav\\n")
        f.write(f"  - Données: {base_name}_data.pkl\\n")
        f.write(f"  - Visualisation: {base_name}_visualization.png\\n")
        f.write(f"  - Rapport: {base_name}_report.txt\\n")
    
    print(f"   📄 Rapport sauvé: {report_path}")
    print(f"✅ Export terminé dans: {output_dir}")
    
    return {
        'audio_path': audio_path,
        'data_path': data_path,
        'viz_path': viz_path,
        'report_path': report_path
    }

def export_dataset_summary(dataset, output_dir='./exports'):
    """Exporte un résumé complet du dataset"""
    
    import os
    from datetime import datetime
    
    os.makedirs(output_dir, exist_ok=True)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    summary_path = os.path.join(output_dir, f"dataset_summary_{timestamp}.csv")
    
    print(f"📊 Export du résumé dataset...")
    
    # Créer un DataFrame avec toutes les statistiques
    summary_data = []
    for i, segment in enumerate(dataset.segments):
        summary_data.append({
            'segment_idx': i,
            'conv_idx': segment.get('conv_idx', -1),
            'start_time': segment.get('start_time', 0),
            'end_time': segment.get('end_time', 0),
            'vad_frames': segment.get('vad_frames', 0),
            'osd_frames': segment.get('osd_frames', 0),
            'vcn_frames': segment.get('vcn_frames', 0),
            'total_frames': int(SEGMENT_DURATION / 0.02),
            'vad_ratio': segment.get('vad_frames', 0) / int(SEGMENT_DURATION / 0.02),
            'osd_ratio': segment.get('osd_frames', 0) / int(SEGMENT_DURATION / 0.02),
            'vcn_ratio': segment.get('vcn_frames', 0) / int(SEGMENT_DURATION / 0.02),
            'has_speech': segment.get('vad_frames', 0) > 0,
            'has_overlap': segment.get('has_overlap', False),
            'has_voice_change': segment.get('has_voice_change', False),
            'num_speakers': len(segment.get('speaker_ids', [])),
            'speakers': str(segment.get('speaker_ids', []))
        })
    
    df_summary = pd.DataFrame(summary_data)
    df_summary.to_csv(summary_path, index=False, encoding='utf-8')
    
    print(f"✅ Résumé sauvé: {summary_path}")
    print(f"   📊 {len(df_summary)} segments exportés")
    
    return df_summary, summary_path

# Exemple d'export
print(f"💾 Fonctions d'export disponibles:")
print(f"   🎯 Segment sélectionné: #{SELECTED_SEGMENT_IDX}")
print(f"   📁 Dossier d'export: ./exports/")

# Décommenter pour exporter le segment sélectionné:
# export_paths = export_segment_analysis(dataset, SELECTED_SEGMENT_IDX)

# Décommenter pour exporter le résumé complet:
# df_summary, summary_path = export_dataset_summary(dataset)

print(f"\\n💡 Décommentez les lignes ci-dessus pour effectuer les exports!")